# Banking Analytics — EDA & KPI Validation

Objective: validate data quality, build customer-aware KPIs, analyze deposits/lending, and prepare evidence for the Power BI dashboard.

Architecture: Data Quality → EDA → Customer / Portfolio Analysis → KPI Validation → Business Questions → Dashboard Handoff.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
df = pd.read_csv(Path('../data/Banking.csv'))
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 1. Data profile and quality

The source contains 3,000 records and 25 columns. Customer-level analysis uses Client ID as the identifier; source-row counts are kept separate.

In [ ]:
source_records = len(df)
distinct_customers = df['Client ID'].nunique()
quality = pd.Series({
    'Source records': source_records,
    'Distinct customers': distinct_customers,
    'Records beyond distinct-customer count': source_records - distinct_customers,
    'Exact duplicate rows': df.duplicated().sum(),
    'Missing Client IDs': df['Client ID'].isna().sum(),
    'Missing income': df['Estimated Income'].isna().sum(),
    'Missing bank loans': df['Bank Loans'].isna().sum(),
    'Missing bank deposits': df['Bank Deposits'].isna().sum()
})
display(quality)
repeated_clients = (df.groupby('Client ID').size().reset_index(name='record_count').query('record_count > 1').sort_values(['record_count','Client ID'], ascending=[False,True]))
print(f'Repeated Client IDs: {len(repeated_clients):,}')
display(repeated_clients.head(10))

In [ ]:
zero_balance = pd.Series({
    'Zero bank-loan records': (df['Bank Loans'] == 0).sum(),
    'Zero bank-deposit records': (df['Bank Deposits'] == 0).sum(),
    'Zero business-lending records': (df['Business Lending'] == 0).sum()
})
display(zero_balance)

## 2. Customer profile and income bands

Income bands are descriptive analytical segments, not external income benchmarks.

In [ ]:
df['Income Band'] = pd.cut(df['Estimated Income'], bins=[-np.inf,100000,250000,np.inf], labels=['Low','Mid','High'], right=False)
income_summary = (df.groupby('Income Band', observed=False).agg(source_records=('Client ID','size'), distinct_customers=('Client ID','nunique'), avg_income=('Estimated Income','mean'), avg_deposits=('Bank Deposits','mean'), avg_loans=('Bank Loans','mean')).reset_index())
display(income_summary)
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='Income Band', order=['Low','Mid','High'])
plt.title('Source Records by Income Band')
plt.tight_layout()
plt.show()

## 3. Portfolio KPIs

These measures describe balances in the source data. They do not support profitability, default/NPL, transaction activity, or credit-card utilization claims.

In [ ]:
portfolio_kpis = pd.Series({
    'Total bank deposits': df['Bank Deposits'].sum(),
    'Total checking accounts': df['Checking Accounts'].sum(),
    'Total saving accounts': df['Saving Accounts'].sum(),
    'Total foreign currency': df['Foreign Currency Account'].sum(),
    'Total bank loans': df['Bank Loans'].sum(),
    'Total business lending': df['Business Lending'].sum(),
    'Average bank deposits / distinct customer': df['Bank Deposits'].sum() / distinct_customers,
    'Loan-to-deposit ratio': df['Bank Loans'].sum() / df['Bank Deposits'].sum()
})
display(portfolio_kpis)
segment_summary = (df.groupby(['Loyalty Classification','Fee Structure']).agg(distinct_customers=('Client ID','nunique'), avg_income=('Estimated Income','mean'), bank_deposits=('Bank Deposits','sum'), bank_loans=('Bank Loans','sum'), business_lending=('Business Lending','sum')).reset_index().sort_values('bank_deposits', ascending=False))
display(segment_summary)

## 4. Balance relationships

Correlation is descriptive and does not establish causation.

In [ ]:
balance_cols = ['Bank Deposits','Checking Accounts','Saving Accounts','Foreign Currency Account','Business Lending','Bank Loans']
corr = df[balance_cols].corr()
plt.figure(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation of Major Banking Balances')
plt.tight_layout()
plt.show()
display(corr)

## 5. KPI validation and dashboard handoff

Customer KPIs: distinct customers, source records, average income, average credit cards, average properties.

Deposit KPIs: bank deposits, checking, savings, foreign currency, average deposits per customer.

Lending KPIs: bank loans, business lending, customers with loans/business lending, loan-to-deposit ratio.

Dashboard views: Executive Overview, Deposit Analysis, Loan Analysis, Customer Relationship.

The central modeling decision is to distinguish Client ID customer counts from raw source-record counts.

In [ ]:
kpi_validation = pd.Series({
    'Distinct Customers': df['Client ID'].nunique(),
    'Source Records': len(df),
    'Repeated Client Records': len(df) - df['Client ID'].nunique(),
    'Average Estimated Income': df['Estimated Income'].mean(),
    'Average Credit Cards': df['Amount of Credit Cards'].mean(),
    'Average Properties Owned': df['Properties Owned'].mean(),
    'Customers with Bank Loans > 0': df.loc[df['Bank Loans'] > 0, 'Client ID'].nunique(),
    'Customers with Business Lending > 0': df.loc[df['Business Lending'] > 0, 'Client ID'].nunique()
})
display(kpi_validation)

## Conclusion

The notebook establishes a customer-aware analytical foundation for the Power BI layer: validated source/customer counts, income segmentation, deposit and lending KPIs, relationship segmentation, and balance correlations. It also documents analytical boundaries where the dataset does not provide sufficient evidence.